In [1]:
import torch
import numpy as np
from pathlib import Path
from torch.profiler import profile, ProfilerActivity, record_function
from typing import Callable, List, Tuple, Dict, Any, Sequence, Union, Optional, Iterable

import sys
sys.path.append("../..")
from v2.model_files.SSD_from_scratch import mySSD as mySSD
from v2.training_files.build_dataloaders import build_train_dl
from v2.training_files.CosSched import build_optimizer_and_scheduler
from v2.training_files.profile_training import profile_SSD_train



device = "cuda" if torch.cuda.is_available() else "cpu"

# desktop, laptop, ubuntu
machine = 'desktop'

# Setup path to data folder
if machine == 'laptop':
    folder_path = Path(r"C:\self-driving-car\data")
elif machine == 'desktop':
    folder_path = Path(r"C:\Udacity_car_data\data")
elif machine == 'ubuntu':
    folder_path = Path(r"/mnt/c/Udacity_car_data/data")

train_path = folder_path / "train"
test_path = folder_path / "test"

In [2]:
ssdmodel = mySSD(class_to_idx_dict={'biker': 0, 'car': 1, 'pedestrian': 2, 'trafficLight': 3, 'truck': 4},
                 in_channels=3,
                 variances=(0.1, 0.2),
                 ).to(device)

train_dataloader, val_dataloader = build_train_dl(train_path=train_path)

optimizer, scheduler = build_optimizer_and_scheduler(model=ssdmodel,
                                                     train_dataloader=train_dataloader,
                                                     max_epochs=150,
                                                     warmup_epochs=5,
                                                     base_lr=0.003,
                                                     min_lr=1e-6,
                                                     momentum=0.9,
                                                     weight_decay=0.005)

scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

In [3]:
profile_SSD_train(model=ssdmodel,
                  train_dataloader=train_dataloader,
                  test_dataloader=val_dataloader,
                  optimizer=optimizer,
                  scheduler=scheduler,
                  scaler=scaler,
                  sched_step_w_opt=False,
                  iou_thresh=0.5,
                  iou_variant="IoU",
                  neg_pos_ratio=3.0,
                  score_thresh=0.5,
                  nms_thresh=0.5,
                  max_detections_per_img=200,
                  epochs=1,
                  device=device)

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                             aten::convolution_backward         2.02%      14.065ms         5.55%      38.651ms     276.076us     167.653ms        44.93%     174.809ms       1.249ms           0 B           0 B       1.36 GB       1.17 G

In [3]:
# after build targets improvement
profile_SSD_train(model=ssdmodel,
                  train_dataloader=train_dataloader,
                  test_dataloader=val_dataloader,
                  optimizer=optimizer,
                  scheduler=scheduler,
                  scaler=scaler,
                  sched_step_w_opt=False,
                  iou_thresh=0.5,
                  iou_variant="IoU",
                  neg_pos_ratio=3.0,
                  score_thresh=0.5,
                  nms_thresh=0.5,
                  max_detections_per_img=200,
                  epochs=1,
                  device=device)

TritonMissing: Cannot find a working triton installation. Either the package is not installed or it is too old. More information on installing Triton can be found at: https://github.com/triton-lang/triton

Set TORCHDYNAMO_VERBOSE=1 for the internal stack trace (please do this especially if you're reporting a bug to PyTorch). For even more developer context, set TORCH_LOGS="+dynamo"
